In [1]:
import os
os.chdir('/home/dlhogan/projects/phd-repos/S3-precipitation-rodeo/')
import glob

# general
import glob
import datetime as dt

# data 
import xarray as xr 
import numpy as np
import pandas as pd

# plotting
import matplotlib.pyplot as plt
import plotly.express as px 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Configure Plotly for Jupyter notebooks
pio.renderers.default = "notebook"
# Alternative renderers you can try if "notebook" doesn't work:
# pio.renderers.default = "plotly_mimetype+notebook"
# pio.renderers.default = "jupyter_lab"

# helper tools
from metpy import calc, units
import scipy.stats as stats
from sklearn.linear_model import LinearRegression

In [5]:
DATA_DIR = '/storage/dlhogan/precipitation-rodeo/data/for_analysis/'
files = glob.glob(os.path.join(DATA_DIR, '*.nc'))
files

['/storage/dlhogan/precipitation-rodeo/data/for_analysis/gothic_precipitation_event_comparisons.nc',
 '/storage/dlhogan/precipitation-rodeo/data/for_analysis/gothic_gridded_precipitation_event_comparisons.nc',
 '/storage/dlhogan/precipitation-rodeo/data/for_analysis/kettle_ponds_precipitation_event_comparisons.nc',
 '/storage/dlhogan/precipitation-rodeo/data/for_analysis/kettle_ponds_gridded_precipitation_event_comparisons.nc']

In [38]:
gothic_prcp_events_obs_ds = xr.open_dataset(files[0]).sel(event_id=slice(0,10))
gothic_prcp_events_gridded_ds = xr.open_dataset(files[1]).sel(event_id=slice(0,10))
kettle_ponds_prcp_events_obs_ds = xr.open_dataset(files[2]).sel(event_id=slice(0,10))
kettle_ponds_prcp_events_gridded_ds = xr.open_dataset(files[3]).sel(event_id=slice(0,10))

Let's plot the recurrence of these events by month, what is their distribution? How many events occur per month?

In [39]:
monthly_counts_list = []
for ins in gothic_prcp_events_obs_ds.sel(benchmark="billy_barr_precip")['test_instrument'].values:
    if ins == 'billy_barr_precip':
        continue
    tmp_ds = gothic_prcp_events_obs_ds.sel(benchmark="billy_barr_precip", test_instrument=ins)
    
    tmp_ds = tmp_ds.assign_coords(
        month=("event_id", tmp_ds["start_time"].dt.month.data)
    )
    grouped = tmp_ds['start_time'].groupby(["test_instrument", "month"])
    monthly_counts = grouped.count()
    monthly_counts_list.append(monthly_counts)
monthly_counts_ds = xr.concat(monthly_counts_list, dim="test_instrument")

In [40]:
# Plot the counts for each instrument by month as a line plot
fig = go.Figure()
for ins in monthly_counts_ds['test_instrument'].values:
    fig.add_trace(go.Scatter(
        x=monthly_counts_ds.sel(test_instrument=ins)['month'],
        y=monthly_counts_ds.sel(test_instrument=ins).values,
        mode='lines+markers',
        name=ins
    ))
fig.update_layout(
    title='Monthly Event Counts by Instrument (Gothic Site)',
    xaxis_title='Month',
    yaxis_title='Number of Events',
    xaxis=dict(tickmode='array', tickvals=list(range(1, 13)), ticktext=[
        'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
        'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
    ])
)
fig.show()

In [41]:
monthly_counts_list = []
for ins in kettle_ponds_prcp_events_obs_ds.sel(benchmark="billy_barr_precip")['test_instrument'].values:
    if ins == 'billy_barr_precip':
        continue
    tmp_ds = kettle_ponds_prcp_events_obs_ds.sel(benchmark="billy_barr_precip", test_instrument=ins)
    
    tmp_ds = tmp_ds.assign_coords(
        month=("event_id", tmp_ds["start_time"].dt.month.data)
    )
    grouped = tmp_ds['start_time'].groupby(["test_instrument", "month"])
    monthly_counts = grouped.count()
    monthly_counts_list.append(monthly_counts)
monthly_counts_ds = xr.concat(monthly_counts_list, dim="test_instrument")

In [42]:
# Plot the counts for each instrument by month as a line plot
fig = go.Figure()
for ins in monthly_counts_ds['test_instrument'].values:
    fig.add_trace(go.Scatter(
        x=monthly_counts_ds.sel(test_instrument=ins)['month'],
        y=monthly_counts_ds.sel(test_instrument=ins).values,
        mode='lines+markers',
        name=ins
    ))
fig.update_layout(
    title='Monthly Event Counts by Instrument (Kettle Ponds Site)',
    xaxis_title='Month',
    yaxis_title='Number of Events',
    xaxis=dict(tickmode='array', tickvals=list(range(1, 13)), ticktext=[
        'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
        'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
    ])
)
fig.show()